# Agent

代理将语言模型与工具结合，创建能够推理任务、决定使用哪些工具，并迭代地工作以寻求解决方案的系统。
create_agent 提供了一个生产就绪的代理实现。
LLM 代理在循环中运行工具以实现目标。代理会一直运行，直到满足停止条件——即模型发出最终输出或达到迭代限制。

## 核心组件

### Model

模型是代理的推理引擎，它可以通过多种方式制定，支持惊呆和动态模型选择

#### Static model

静态模型是在创建代理时配置一次，并在整个执行过翅中保持不变

In [5]:
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse, dynamic_prompt, AgentMiddleware
from langchain.agents.middleware.types import StateT
from langchain.agents.structured_output import ToolStrategy
from langchain_core.messages import ToolMessage
from langchain_deepseek import ChatDeepSeek
from langgraph.runtime import Runtime
from langgraph.typing import ContextT
from pydantic import BaseModel

agent = create_agent(
    "deepseek-chat",
    tools=[]
)

# 更详细的参数设置

model = ChatDeepSeek(
    model="deepseek-chat",
    temperature=0.1,
    max_tokens=1000,
    timeout=50
)

#### 动态模型

动态模型是运行时基于当前状态以及上下文实现复杂的路由逻辑和成本控制。
要使用动态模型，使用@wrao_model_call装饰器创建中间件，该中间件会在请求中修改模型

In [3]:
base_model = ChatDeepSeek(model="deepseek-chat")

advanced_model = ChatDeepSeek(model="deepseek-reasoner")


@wrap_model_call
def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:
    """choose model based on conversation complexity."""
    message_content = len(request.state["messages"])

    if message_content > 10:
        model = advanced_model
    else:
        model = base_model

    request.model = model
    return handler(request)


agent = create_agent(
    model=base_model,
    tools=[],
    middleware=[dynamic_model_selection]
)

## Tools

工具使代理能够执行操作，智能体不仅仅局限于单一模型工具绑定，而是通过以下方式实现更强大的功能：
* 连续多次调用工具
* 在适当情况下并行调用工具
* 根据先前结果动态选择工具
* 工具重试逻辑和错误处理
* 跨工具调用的状态持久化


### 定义工具

将工具列表传递给代理


In [ ]:
from langchain_core.tools import tool


@tool
def search(query: str) -> str:
    """Search for information."""
    return f"Results for {query}"


@tool
def get_weather(location: str) -> str:
    """Get weather information for a location."""
    return f"Weather at {location}"


agent = create_agent(
    model,
    tools=[get_weather, search]
)

### 工具错误处理

要自定义如何处理工具错误，使用@wrap_tool_call装饰器来创建中间件

In [ ]:
@wrap_model_call
def handle_tool_errors(request, handler):
    """Handle tool execution errors with custom messages"""
    try:
        return handler(request)
    except Exception as e:
        return ToolMessage(
            content=f"Tool error: Please check your input and try again.({str(e)})",
        )


agent = create_agent(
    model,
    tools=[handle_tool_errors]
)

### ReAct循环中的工具使用

代理遵循ReAct("推理+行动")模式，在简短的图例步骤和有针对性的工具调用之间交替，并将生成的规模观察结果输入后续决策，直到能够提供最终答案

### 动态系统提示

对于徐璈根据运行时上下文或代理状态修改系统提示的高级用例，可以使用中间件
@dynamic_prompt 装饰器创建中间件，根据模型请求动态生成系统提示

In [ ]:
from typing import TypedDict, Any


class Context(TypedDict):
    user_role: str


@dynamic_prompt
def user_role_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on user role"""
    user_role = request.runtime.context.get("user_role", "user")
    base_prompt = "You are a helpful assistant."

    if user_role == "expert":
        return f"{base_prompt} Provide detailed technical responses."
    elif user_role == "beginner":
        return f"{base_prompt} Explain concepts simply and avoid jargon."

    return base_prompt


agent = create_agent(
    model,
    tools=[],
    middleware=[user_role_prompt],
    context_schema=Context
)

result = agent.invoke(
    {
        "message": [{"role": "user", "content": "Explain machine learning"}],

    },
    context={"user_role": "expert"}
)

### 调用

可用通过向State发送更新来调用一个代理。所有代理在其状态中都包含一系列消息；要调用代理，需要传递一条新的消息

In [ ]:
result = agent.invoke(
    {"message": [{"role": "user", "content": "What's th weather in San Francisco?"}]}
)

## 高级概念

### 结构化输出
在某些情况下，代理可以以特定格式输出返回，LangChain通过response_format 参数通了结构化输出的策略。

#### 工具策略

ToolStrategy 使用人工工具调用生成结构化输出。这适用于任何支持工具调用的模型

In [7]:
class ContactInfo(BaseModel):
    name: str
    email: str
    phone: str


agent = create_agent(
    model,
    tools=[],
    response_format=ToolStrategy(ContactInfo)
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: john Doe, 123@example.com"}],
})

result["structured_response"]

ContactInfo(name='john Doe', email='123@example.com', phone='')

#### ProviderStrategy 提供者策略

ProviderStrategy 使用模型提供者的原生结构化输出生成，这更可靠，但仅适用于支持原生结构化输出的提供者。

### Memory 记忆

代理通过消息状态自动维护对话历史，还可以通过配置代理使用自定义状态码模式，在对话过程中记住额外信息。
状态中存储的信息可以被视为代理的短期记忆
自动以状态模式必须扩展AgentState作为TypedDict
定义自定义状态有两种方法：
1. 通过中间件（推荐）
2. 通过state_schema在create_agent上定义

#### 通过中间件定义状态

使用中间件来定义状态，当你的自定义状态需要被附加到该中间件的特定中间件钩子和工具访问时。

In [ ]:
class CustomState(AgentState):
    user_preferences: dict


class CustomMiddleware(AgentMiddleware):
    state_schema = CustomState
    tools = []

    def before_model(self, state: StateT, runtime: Runtime[ContextT]) -> dict[str, Any] | None:
        ...


agent = create_agent(
    model,
    tools=[],
    middleware=[CustomMiddleware]
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Explain machine learning"}],
    "user_preferences": {"style": "technical", "verbosity": "detailed"},
})

#### 通过state_schema定义状态

使用state_schema 参数作为快捷方式来定义仅在工具中使用的自定义状态

In [ ]:

class CustomState(AgentState):
    user_preferences: dict


agent = create_agent(
    model,
    tools=[],
    middleware=CustomState
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "I prefer technical explanations"}],
    "user_preferences": {"style": "technical", "verbosity": "detailed"},
})

## 流式传输

可以使用invoke调用代理以获取最终响应，如果执行多个步骤，为了展示中间步骤，我们可以随着消息发生而流式返回消息

In [ ]:
for chunk in agent.stream({
    "messages": [{"role": "user", "content": "Explain machine learning"}],
}, stream_mode="values"):
    latest_message = chunk["messages"][-1]
    if latest_message.content:
        print(f"Agent:{latest_message.content}")
    elif latest_message.tool_calls:
        print(f"Calling tools:{[tc['name'] for tc in latest_message.tool_calls]}")


## Middleware 中间件

中间件为自定义执行阶段中的代理行为提供了强大的拓展性
* 在模型调用之前处理状态（比如：消息修剪、上下文注入）
* 修改或验证模型的响应（比如: 安全护栏、内容过滤）
* 使用自定义逻辑处理工具执行错误
* 根据状态或上下文实现动态模型选择
* 添加自定义日志记录、监控或分析

中间件可以无缝集成到代理的执行图中，允许在关键点拦截和修改数据流，无须担心更改核心代码逻辑